In [1]:
import run as run
import chephy_model as cpm
import GNN_model
import gnn_utils
import numpy as np
import torch
import pandas as pd


/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def create_hetero_data(input_file, label_header, cdr3_header, atchley_dict, n=1000, distance_threshold = 0.2, use_mst=True, enrich=False):

    data = run.load_dataframe(input_file)
    data = data.sample(n=n, random_state=42)

    data = run.prepare_data(data, cdr3_header, label_header)    
    data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

    sequences = set(data["cdr3_truncated"])
    epitopes = list(set(data[label_header]))

    chephy_matrix, sequences_list = cpm.find_che_phy_dist(sequences)
    sequence_indices = np.arange(len(sequences_list))

    data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
    data['epitope_index'] = data[label_header].apply(lambda x: epitopes.index(x) if x in epitopes else -1)
    print("Done indexing")

    data_hetero, labels = gnn_utils.build_hetero_graph(
                            data,
                            chephy_matrix,
                            sequences_list,
                            epitopes,
                            atchley_dict,
                            distance_threshold=distance_threshold,
                            use_mst=use_mst,
                            enrich=enrich  
                        )
    return data_hetero, data


In [3]:
atchley_path = "/home/dsi/chenbis/repos/sol_lab/files/atchley.csv"
atchley_df = pd.read_csv(atchley_path)
atchley_dict = atchley_df.set_index("amino.acid").to_dict(orient="index")

In [4]:
input_file = "/home/dsi/chenbis/repos/sol_lab/data/train.tsv"
label_header = "peptide"
cdr3_header = "cdr3"
n = 30000
distance_threshold=0.01
use_mst=True
enrich=True

data_hetero_train, data_train = create_hetero_data(input_file, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)

sequence_vectors done on GPU
pdist done on GPU
normalized done on GPU
moved to CPU
Done indexing
computing MST
MST done
enrichment
Step 1, Done!
Step 2, Done!
Step 3, Done!
Step 4, Done!
Step 5, Done!


In [ ]:
input_file = "/work/test.tsv"
label_header = "peptide"
cdr3_header = "cdr3"
n = 10000
distance_threshold=0.01
use_mst=False
enrich=True

data_hetero_test, data_test = create_hetero_data(input_file, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GNN_model.HeteroTCR(data_hetero_train.metadata(), 1024, 3, "SAGE")
model = model.to(device)
data_hetero_train = data_hetero_train.to(device)
data_hetero_test = data_hetero_test.to(device)

In [ ]:
# Create edge label indices for training data
train_edge_pairs = data_train[['tcr_index', 'epitope_index']].values
train_edge_label_index = torch.tensor(train_edge_pairs.T)

# Create edge label indices for test data
test_edge_pairs = data_test[['tcr_index', 'epitope_index']].values
test_edge_label_index = torch.tensor(test_edge_pairs.T)

# Create labels
y_train = data_train['Binding'].values
y_test = data_test['Binding'].values

In [ ]:
import os

save_model_path = '/work/models'
NUM_EPOCHS=1000

epoch = []
loss = []
acc = []
auc_roc = []
val_loss = []
val_acc = []
val_auc_roc = []
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.0)

for ep in range(1, NUM_EPOCHS+1):
    train_loss, train_binary_accuracy, train_ROCAUC, valid_loss, val_binary_accuracy, val_ROCAUC = GNN_model.train(model, optimizer, 
                                                                                                                   data_hetero_train, data_hetero_test, 
                                                                                                                   train_edge_label_index, test_edge_label_index, 
                                                                                                                   y_train, y_test, 
                                                                                                                   device)
    
    print(
    'Train epoch: {} - loss: {:.4f} - binary_accuracy: {:.4f} - ROCAUC: {:.4f} - val_loss: {:.4f} - val_binary_accuracy: {:.4f} - val_ROCAUC: {:.4f}'.format(
    ep, train_loss.item(), train_binary_accuracy, train_ROCAUC, valid_loss, val_binary_accuracy, val_ROCAUC))
    torch.save(model.state_dict(), os.path.join(save_model_path, 'HeteroTCR_epoch{:03d}_AUC{:.6f}.pth'.format(ep, val_ROCAUC.item())))
    
    loss.append(train_loss.item())
    acc.append(train_binary_accuracy.item())
    auc_roc.append(train_ROCAUC.item())
    val_loss.append(valid_loss.item())
    val_acc.append(val_binary_accuracy.item())
    val_auc_roc.append(val_ROCAUC.item())
    epoch.append(ep)

In [ ]:
root_history = 'work'
# Logs of results
dfhistory = {'epoch': epoch,
            'loss': loss, 'acc': acc, 'auc_roc': auc_roc,
            'val_loss': val_loss, 'val_acc': val_acc, 'val_auc_roc': val_auc_roc}
df = pd.DataFrame(dfhistory)
if not os.path.exists(root_history):
    os.makedirs(root_history)
df.to_csv("work/history.csv", header=True, index=False)

# get num_epoch
val_auc_roc = list(df['val_auc_roc'])
max_auc_index = val_auc_roc.index(max(val_auc_roc))
num_epoch = max_auc_index + 1
print("max auc epoch:", num_epoch)
print(max(val_auc_roc))

val_loss = list(df['val_loss'])
min_val_loss_index = val_loss.index(min(val_loss))
loss_num_epoch = min_val_loss_index + 1
print("loss auc epoch:", loss_num_epoch)
print(val_auc_roc[min_val_loss_index])

for root, dirs, files in os.walk(save_model_path):
    for file in files:
        if file.startswith("HeteroTCR_epoch{:03d}".format(loss_num_epoch)):
            os.rename(os.path.join(save_model_path, file), os.path.join(save_model_path, 'minloss_AUC_' + file))
        elif file.startswith("HeteroTCR_epoch{:03d}".format(num_epoch)):
            os.rename(os.path.join(save_model_path, file), os.path.join(save_model_path, 'max_AUC_' + file))
        else:
            os.remove(os.path.join(save_model_path, file))

In [ ]:
# Read McPAS-TCR data
mcpas_df = pd.read_csv('/work/McPAS-TCR.csv')
# mcpas_df = pd.read_csv('/work/McPAS-TCR.csv')


# Select and rename columns from McPAS
mcpas_subset = mcpas_df[['CDR3.beta.aa', 'Epitope.peptide']].copy()
mcpas_subset.columns = ['cdr3', 'peptide']
mcpas_subset['Binding'] = 1  # Positive samples

# Drop any rows with NaN values
mcpas_subset = mcpas_subset.dropna()

# Read negative samples
neg_df = pd.read_csv('/work/vdjdb_score321_with_negatives.csv')

# Select and rename columns from negative samples
neg_subset = neg_df[neg_df['true.label'] == 0][['cdr3', 'antigen.epitope']].copy()
neg_subset.columns = ['cdr3', 'peptide']
neg_subset['Binding'] = 0  # Negative samples

# Drop any rows with NaN values
neg_subset = neg_subset.dropna()

# Get existing CDR3s and peptides from train and test data to exclude
existing_pairs = set(zip(data_train['cdr3'], data_train['peptide'])) | set(zip(data_test['cdr3'], data_test['peptide']))

# Remove rows that exist in train or test data
mcpas_subset = mcpas_subset[~mcpas_subset.apply(lambda x: (x['cdr3'], x['peptide']) in existing_pairs, axis=1)]
neg_subset = neg_subset[~neg_subset.apply(lambda x: (x['cdr3'], x['peptide']) in existing_pairs, axis=1)]

# Sample 1000 rows from each dataset
mcpas_sample = mcpas_subset.sample(n=2000, random_state=42)
neg_sample = neg_subset.sample(n=2000, random_state=42)

# Combine the samples
combined_df = pd.concat([mcpas_sample, neg_sample], ignore_index=True)

# Shuffle the final dataset
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
import sol_lab.pycode.run as run


cdr3_header = "cdr3"
label_header = "peptide"

combined_df = run.prepare_data(combined_df, cdr3_header, label_header)    
combined_df = truncate_sequences(combined_df, cdr3_header, 4, 4)
combined_df["cdr3_truncated"] = combined_df["cdr3_truncated"].str.upper()
sequences = set(combined_df["cdr3_truncated"])
epitopes = list(set(combined_df[label_header]))

chephy_matrix, sequences_list = find_che_phy_dist(sequences)
# todo: add option to filter sequences by hamming distance before create data hetero.

combined_df['tcr_index'] = combined_df['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
combined_df['epitope_index'] = combined_df[label_header].apply(lambda x: epitopes.index(x) if x in epitopes else -1)

sequence_indices = np.arange(len(sequences_list))



distance_threshold=0.01
hamming_threshold=8
use_mst=False
enrich=True

combined_hetero, tcr_lables = build_hetero_graph(
    combined_df,
    chephy_matrix,
    sequences_list,
    epitopes,
    atchley_dict,
    distance_threshold=0.01,
    use_mst=False,
    enrich=True
)

# combined_hetero = build_hetero_graph(combined_df, chephy_matrix, sequences_list, epitopes, atchley_dict, distance_threshold=0.2)

# Create edge label indices for test data
combined_edge_pairs = combined_df[['tcr_index', 'epitope_index']].values
combined_edge_label_index = torch.tensor(combined_edge_pairs.T)

# Create labels
y_combined = combined_df['Binding'].values


In [ ]:
model = HeteroTCR(combined_hetero.metadata(), 1024, 3, "SAGE")
model = model.to(device)
combined_hetero = combined_hetero.to(device)

with torch.no_grad():  # Initialize lazy modules.
    out = model(combined_hetero.x_dict, combined_hetero.edge_index_dict, combined_edge_label_index)

model_dir = "/work/models"
val_model_path = "/work/models"
for root, dirs, files in os.walk(val_model_path):
    for file in files:
        if file.startswith("min"):
            PATH = os.path.join(val_model_path, file)
            model.load_state_dict(torch.load(PATH))
            test_loss, test_binary_accuracy, test_ROCAUC, test_prob = predict(model, combined_hetero, combined_edge_label_index, y_combined)
            print("ACC: {:.4f}".format(test_binary_accuracy))
            print("AUC: {:.4f}".format(test_ROCAUC))

In [ ]:
model = HeteroTCR(data_hetero_test.metadata(), 1024, 3, "SAGE")
model = model.to(device)
data_hetero_test = data_hetero_test.to(device)

with torch.no_grad():  # Initialize lazy modules.
    out = model(data_hetero_test.x_dict, data_hetero_test.edge_index_dict, test_edge_label_index)

model_dir = "/work/models"
val_model_path = "/work/models"
for root, dirs, files in os.walk(val_model_path):
    for file in files:
        if file.startswith("min"):
            PATH = os.path.join(val_model_path, file)
            model.load_state_dict(torch.load(PATH))
            test_loss, test_binary_accuracy, test_ROCAUC, test_prob = predict(model, data_hetero_test, test_edge_label_index, y_test)
            print("ACC: {:.4f}".format(test_binary_accuracy))
            print("AUC: {:.4f}".format(test_ROCAUC))